In [0]:
from pyspark.sql.functions import sum, round

df = spark.table("workspace.default.online_retail_ii")

clean_df = (
    df
    .dropna(subset=["Customer ID"])
    .withColumn(
        "Revenue",
        round(df.Quantity * df.Price, 2)
    )
)

In [0]:
top_customers = (
    clean_df
    .groupBy("Customer ID")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
    .limit(10)
)

top_customers.show()

In [0]:
from pyspark.sql.functions import count

order_count = (
    clean_df
    .groupBy("Customer ID")
    .agg(
        count("Invoice").alias("OrderCount") 
    )
    .orderBy("OrderCount", ascending=False)
    .limit(10)
)

order_count.show()

In [0]:
clean_df.show()

In [0]:
from pyspark.sql.functions import avg

customer_avg_revenue = (
    clean_df
    .groupBy("Customer ID")
    .agg(
    avg("Revenue").alias("AverageRevenue")
    )
    .orderBy("AverageRevenue", ascending=False)
    .limit(10)
)

In [0]:
customer_total_revenue = (
    clean_df
    .groupBy("Customer ID")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
)

vip_customers = (
    customer_total_revenue
    .filter("TotalRevenue > 10000")
    .select("Customer ID","TotalRevenue")
)

vip_customers.show()

In [0]:
clean_df = clean_df.withColumnRenamed("Customer ID", "CustomerID")

clean_df.write.mode("overwrite").saveAsTable("clean_sales")